# Tutorial 10 — Learning and testing the pairwise couple

The classical state-space model is the special case of the Gaussian **pairwise Markov
model** with no *measurement back-action* (`A_xy = 0`) and no *observation memory*
(`A_yy = 0`). This notebook shows that these two couple-defining coefficients can be

1. **recovered** from data by a *partial* EM (the smoother is the E-step), starting
   from the classical initialisation `A_xy = A_yy = 0`; and
2. **tested** — a likelihood-ratio test decides whether back-action is present at all.

It reproduces, in miniature, Figs. 2–3 of the paper *Smoothing, Learning, and Testing
the Gaussian Pairwise Markov Model* (Sec. IV). The full calibrated study is in
`experiments/em_identification.py` and `experiments/em_lrt.py`.

In [ ]:
import sys
from pathlib import Path

# Make `prg` importable from the notebooks directory
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from prg.classes.linear_pkf import Linear_PKF
from prg.classes.param_linear import ParamLinear
from prg.models.linear._amq import LinearAmQ
from prg.learning.em_partial_dynamics import estimate_dynamics_em, back_action_lrt

Q_TRUE = np.array([[0.10, 0.05], [0.05, 0.10]])   # correlated process noise (R_xy != 0)
AXX, AYX = 0.6, 0.3                                # blocks a classical model also has

def couple_param(A_xy, A_yy, Q=Q_TRUE):
    """A scalar linear pairwise model with the given back-action / obs-memory.

    Built through LinearAmQ so the transition *callables* match the A *matrix* —
    overwriting the A attribute of a factory model would leave the simulator on its
    default transition."""
    A = np.array([[AXX, A_xy], [AYX, A_yy]])
    m = LinearAmQ(1, 1, A=A, mQ=0.5 * (Q + Q.T) + 1e-9 * np.eye(2),
                  mz0=np.zeros((2, 1)), Pz0=np.eye(2), pairwiseModel=True)
    kw = m.get_params().copy(); kw.pop("dim_x"); kw.pop("dim_y")
    return ParamLinear(0, 1, 1, **kw)

CLASSICAL_INIT = couple_param(0.0, 0.0)            # A_xy = A_yy = 0

## 1. A couple with back-action

We simulate a scalar pairwise process whose latent state is partly driven by the
observed channel (`A_xy = 0.4`) and whose observation carries memory (`A_yy = 0.4`),
with correlated process noise. A classical model cannot express either effect.

In [ ]:
true = couple_param(0.4, 0.4)
data = Linear_PKF(true, sKey=0).simulate_N_data(1500)
print(f"simulated {len(data)} steps; the classical model would force A_xy = A_yy = 0")

## 2. Recovering the coupling by partial EM

`estimate_dynamics_em` holds `A_xx`, `A_yx`, `Q` fixed and learns `A_xy`, `A_yy` from
the classical initialisation. Each E-step is one variational (`VAR`) smoothing pass;
the M-step is a closed-form regression of the residual on the observed `y`. The
observed-data log-likelihood increases monotonically.

In [ ]:
res = estimate_dynamics_em(CLASSICAL_INIT, data, tol=1e-5, max_iter=200)
print(f"recovered  A_xy = {res.A_xy.item():.3f}   A_yy = {res.A_yy.item():.3f}   "
      f"(true 0.4 / 0.4;  converged in {res.n_iter} iters)")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(res.loglik, "o-", ms=3)
ax.set(xlabel="EM iteration", ylabel="observed-data log-likelihood",
       title="Partial EM climbs out of the classical model (monotone)")
fig.tight_layout(); plt.show()

## 3. Identifiability — `A_yy` recovers more tightly than `A_xy`

The latent state carries a gauge freedom (`X -> T X`). The observation memory `A_yy`
couples *observed* coordinates and is gauge-invariant, hence fully identifiable; the
back-action `A_xy` is identifiable only up to the gauge (pinned here by fixing `A_xx`,
`Q`). So across seeds, `A_yy` concentrates more tightly than `A_xy`.

In [ ]:
axy, ayy = [], []
for s in range(6):
    d = Linear_PKF(couple_param(0.4, 0.4), sKey=s).simulate_N_data(1500)
    r = estimate_dynamics_em(CLASSICAL_INIT, d, tol=1e-5, max_iter=200)
    axy.append(r.A_xy.item()); ayy.append(r.A_yy.item())
axy, ayy = np.array(axy), np.array(ayy)
print(f"A_xy = {axy.mean():.3f} +/- {axy.std():.3f}   (latent-mediated, weaker)")
print(f"A_yy = {ayy.mean():.3f} +/- {ayy.std():.3f}   (gauge-invariant, tighter)")

fig, ax = plt.subplots(figsize=(6, 3))
ax.axhline(0.4, ls=":", color="k", lw=1, label="truth (0.4)")
ax.scatter(np.zeros_like(axy), axy, c="tab:red", label="A_xy (back-action)")
ax.scatter(np.ones_like(ayy), ayy, c="tab:green", label="A_yy (obs. memory)")
ax.set_xticks([0, 1]); ax.set_xticklabels(["A_xy", "A_yy"])
ax.set(ylabel="estimate over 6 seeds",
       title="A_yy is recovered more tightly than A_xy")
ax.legend(fontsize=8); fig.tight_layout(); plt.show()

## 4. Testing for back-action (likelihood-ratio test)

Is back-action present at all? Test `H0: A_xy = 0` against `A_xy != 0`. Both models
are fit by EM (`A_yy` a free nuisance in each); the statistic
`Lambda = 2[ell(free) - ell(A_xy=0)]` is asymptotically chi-square with
`dim_x * dim_y` degrees of freedom under `H0`.

In [ ]:
data_ba = Linear_PKF(couple_param(0.4, 0.4), sKey=0).simulate_N_data(1500)  # back-action present
data_no = Linear_PKF(couple_param(0.0, 0.4), sKey=0).simulate_N_data(1500)  # back-action absent

lrt_ba = back_action_lrt(CLASSICAL_INIT, data_ba, tol=1e-5)
lrt_no = back_action_lrt(CLASSICAL_INIT, data_no, tol=1e-5)
print(f"back-action present : Lambda = {lrt_ba.stat:8.2f}   p = {lrt_ba.pvalue:.1e}   -> reject H0")
print(f"back-action absent  : Lambda = {lrt_no.stat:8.2f}   p = {lrt_no.pvalue:.3f}     -> keep H0")
print(f"(chi-square_{lrt_ba.dof} critical value at 5% = 3.84)")

## Going further

- **`experiments/em_identification.py`**, **`experiments/em_lrt.py`** — the full,
  calibrated study behind Figs. 2–3 (50 seeds for recovery; 350 seeds for the
  chi-square_1 null and the power curve; empirical test size 0.049 at alpha = 0.05).
- **`prg.learning.em_partial_dynamics`** — `estimate_dynamics_em` and
  `back_action_lrt` used here. **`prg.learning.em_partial_noise`** learns the noise
  block `Q` (with `A` fixed) instead.
- Tutorials **07** and **09** cover the six linear smoothers that supply the E-step.